In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib captum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 6.6 MB/s eta 0:00:00


In [9]:
import numpy as np
import os
import pandas as pd
from sklearn.metrics import recall_score, f1_score, accuracy_score

from scipy.special import softmax

In [10]:
# Cell 2 — mount Drive (μόνο αυτό χρειάζεται από Colab-specific)
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = "/content/drive/MyDrive/Colab Notebooks/SER_Thesis/30_cross_corpus_analysis/results"
CORPORA = ["TESS", "RAVDESS", "CREMA-D", "IEMOCAP"]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# Τρεις άδειοι πίνακες 4×4, DataFrame, με labels στις γραμμές/στήλες

uar_matrix = pd.DataFrame(index=CORPORA, columns=CORPORA, dtype=float)
f1_matrix = pd.DataFrame(index=CORPORA, columns=CORPORA, dtype=float)
acc_matrix = pd.DataFrame(index=CORPORA, columns=CORPORA, dtype=float)

In [12]:
#γραμμή: corpus που εκπαιδεύτηκε, στήλη: corpus που έγινε evaluation
#διαγώνιος: πάντα in-domain απόδοση, εκτός διαγωνίου cross-corpus απόδοση

for source in CORPORA:
    for target in CORPORA:
        path = os.path.join(RESULTS_DIR, f"{source}_to_{target}.npz")
        data = np.load(path)
        preds = data["predictions"] #προβλέψεις του source μοντέλου πάνω στο test set του target
        gt = data["ground_truth"]   #πραγματικές ετικέτες (common_label_id) του target μοντέλου

        # UAR = unweighted average recall
        uar = recall_score(gt, preds, average="macro", zero_division=0)
        #Macro F1
        f1 = f1_score(gt, preds, average="macro", zero_division=0)
        #Απλή accuracy (συμπληρωματικά λόγικα ανισσοροπίας κλάσεων)
        acc = accuracy_score(gt, preds)

        uar_matrix.loc[source, target] = uar
        f1_matrix.loc[source, target] = f1
        acc_matrix.loc[source, target] = acc

print("=== UAR (πηγή=γραμμή, στόχος=στήλη) ===")
print(uar_matrix.round(3))
print("\n=== Macro F1 ===")
print(f1_matrix.round(3))
print("\n=== Accuracy ===")
print(acc_matrix.round(3))

=== UAR (πηγή=γραμμή, στόχος=στήλη) ===
          TESS  RAVDESS  CREMA-D  IEMOCAP
TESS     0.566    0.330    0.269    0.275
RAVDESS  0.260    0.660    0.520    0.345
CREMA-D  0.339    0.576    0.790    0.431
IEMOCAP  0.295    0.271    0.453    0.664

=== Macro F1 ===
          TESS  RAVDESS  CREMA-D  IEMOCAP
TESS     0.518    0.272    0.240    0.174
RAVDESS  0.139    0.599    0.517    0.275
CREMA-D  0.248    0.536    0.784    0.365
IEMOCAP  0.183    0.153    0.382    0.654

=== Accuracy ===
          TESS  RAVDESS  CREMA-D  IEMOCAP
TESS     0.566    0.286    0.266    0.192
RAVDESS  0.260    0.619    0.523    0.318
CREMA-D  0.339    0.587    0.787    0.427
IEMOCAP  0.295    0.306    0.464    0.642


In [13]:
# --- Worst-corpus performance ανά source model ---
worst_case = uar_matrix.min(axis=1)
print("\nWorst-corpus UAR ανά source model:")
print(worst_case.round(3))


Worst-corpus UAR ανά source model:
TESS       0.269
RAVDESS    0.260
CREMA-D    0.339
IEMOCAP    0.271
dtype: float64


In [14]:
# --- Cross-corpus drop: in-domain (διαγώνιος) vs μέσος όρος cross-corpus ---

#Εξαγωγή της διαγωνίου
diag = pd.Series(np.diag(uar_matrix), index=CORPORA)

#Μέσος όρος UAR για τα υπόλοιπα στοιχεία κάθε γραμμής
off_diag_mean = pd.Series({
    s: uar_matrix.loc[s, [t for t in CORPORA if t != s]].mean() for s in CORPORA
})

#Διαφορά στοιχείων διαγωνιου με τα υπόλοιπα
drop = diag - off_diag_mean
print("\nIn-domain UAR:")
print(diag.round(3))
print("\nΜέσο cross-corpus UAR:")
print(off_diag_mean.round(3))
print("\nCross-corpus drop (in-domain - cross-corpus):")
print(drop.round(3))


In-domain UAR:
TESS       0.566
RAVDESS    0.660
CREMA-D    0.790
IEMOCAP    0.664
dtype: float64

Μέσο cross-corpus UAR:
TESS       0.291
RAVDESS    0.375
CREMA-D    0.449
IEMOCAP    0.340
dtype: float64

Cross-corpus drop (in-domain - cross-corpus):
TESS       0.275
RAVDESS    0.285
CREMA-D    0.341
IEMOCAP    0.324
dtype: float64


In [15]:
# --- Ο δικός σου ECE (equal-width bins), αυτούσιος ---
def expected_calibration_error(confidences, correct, n_bins=15):
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_info = []
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        in_bin = (confidences > lo) & (confidences <= hi) if i > 0 else (confidences >= lo) & (confidences <= hi)
        prop = np.mean(in_bin)
        if prop > 0:
            acc_bin = np.mean(correct[in_bin])
            conf_bin = np.mean(confidences[in_bin])
            ece += prop * abs(acc_bin - conf_bin)
            bin_info.append((lo, hi, acc_bin, conf_bin, prop))
    return ece, bin_info

In [16]:
# --- Πίνακας 4×4 ECE, ίδια δομή/σύμβαση με το uar_matrix (γραμμή=πηγή, στήλη=στόχος) ---
ece_matrix = pd.DataFrame(index=CORPORA, columns=CORPORA, dtype=float)

for source in CORPORA:
    for target in CORPORA:
        path = os.path.join(RESULTS_DIR, f"{source}_to_{target}.npz")
        data = np.load(path)
        preds = data["predictions"]
        gt = data["ground_truth"]
        logits = data["logits"]           # (n_samples, 4), ήδη περιορισμένα στις κοινές κλάσεις

        probs = softmax(logits, axis=1)
        confidences = probs.max(axis=1)   # μέγιστη πιθανότητα = "confidence" της πρόβλεψης
        correct = (preds == gt).astype(int)

        ece, _ = expected_calibration_error(confidences, correct, n_bins=15)
        ece_matrix.loc[source, target] = ece

print("=== ECE (πηγή=γραμμή, στόχος=στήλη) ===")
print(ece_matrix.round(3))

=== ECE (πηγή=γραμμή, στόχος=στήλη) ===
          TESS  RAVDESS  CREMA-D  IEMOCAP
TESS     0.219    0.375    0.379    0.498
RAVDESS  0.337    0.184    0.079    0.215
CREMA-D  0.391    0.229    0.079    0.325
IEMOCAP  0.542    0.593    0.247    0.077


In [17]:
# Worst-corpus calibration: ΜΕΓΙΣΤΟ ECE ανά πηγή (το χειρότερο-βαθμονομημένο target)
worst_case_ece = ece_matrix.max(axis=1)
print("\nWorst-corpus ECE ανά source model (μέγιστο = χειρότερο):")
print(worst_case_ece.round(3))

# In-domain ECE (διαγώνιος)
diag_ece = pd.Series(np.diag(ece_matrix), index=CORPORA)

# Μέσο cross-corpus ECE (εκτός διαγωνίου)
off_diag_mean_ece = pd.Series({
    s: ece_matrix.loc[s, [t for t in CORPORA if t != s]].mean() for s in CORPORA
})

# Calibration shift = cross-corpus - in-domain (θετικό = χειροτέρευση βαθμονόμησης εκτός corpus)
calibration_shift = off_diag_mean_ece - diag_ece

print("\nIn-domain ECE:")
print(diag_ece.round(3))
print("\nΜέσο cross-corpus ECE:")
print(off_diag_mean_ece.round(3))
print("\nCalibration shift (cross-corpus - in-domain):")
print(calibration_shift.round(3))


Worst-corpus ECE ανά source model (μέγιστο = χειρότερο):
TESS       0.498
RAVDESS    0.337
CREMA-D    0.391
IEMOCAP    0.593
dtype: float64

In-domain ECE:
TESS       0.219
RAVDESS    0.184
CREMA-D    0.079
IEMOCAP    0.077
dtype: float64

Μέσο cross-corpus ECE:
TESS       0.417
RAVDESS    0.210
CREMA-D    0.315
IEMOCAP    0.460
dtype: float64

Calibration shift (cross-corpus - in-domain):
TESS       0.199
RAVDESS    0.026
CREMA-D    0.235
IEMOCAP    0.384
dtype: float64
